# Figure 4: cross-cell-line transfer and context diagnostics

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Data preparation, training, and loading provenance

Panel a summarizes all K562 target perturbations. Panels b-e stratify
response quality by the number of source cell lines in which the
perturbation was observed. Panel f changes the prior-interaction background while holding the target K562 context and
the 506 source-coverage-0 perturbations fixed. Training and inference are registered under `XCL-MAIN`;
STATE, TxPert, and diagnostic outputs are loaded from archived tables.

In [ ]:
panel_map = REGISTRY.loc[REGISTRY["figure"].eq("Fig4")].copy()
required = ["source_data", "training_code", "evaluation_code", "plot_code", "canonical_panel"]
display(panel_map[["panel", "panel_type", "experiment_id", "claim_or_role", "status"]])

def archived_paths_exist(value, base):
    if value == "NA":
        return True
    return all((base / item).exists() for item in str(value).split(";"))

for column in ["source_data", "plot_code", "canonical_panel"]:
    missing = [
        value for value in panel_map[column]
        if not archived_paths_exist(value, ARCHIVE)
    ]
    assert not missing, f"Missing {column}: {missing}"
print("Panel-level figure inputs and plotting assets are present.")

In [ ]:
source = SOURCE_DATA / "Fig4"
display(pd.read_csv(source / "source_data_overall_mean_metrics.csv"))
coverage = pd.read_csv(source / "source_data_panels_b_e.csv")
display(coverage[[
    "source_context_count", "n_conditions",
    "delta_pcc_mean_across_seed_mean",
    "topk_de_overlap_mean_across_seed_mean",
    "pra_top100_mean_across_seed_mean",
    "csa_mean_across_seed_mean",
]])

In [ ]:
for script in [
    "make_cross_cell_overall_mean_bar_20260720.py",
    "make_cross_cell_remaining_panels_20260718.py",
    "make_interaction_context_ablation_fig4f_20260729.py",
]:
    subprocess.run(
        [sys.executable, str(ARCHIVE / "scripts" / "Fig4" / script)],
        check=True,
    )

In [ ]:
outputs = [
    ("a", "Overall mean-response recovery", REPRO / "Fig4" / "figure4a_cross_cell_overall_mean_metrics.svg"),
    ("b", "Perturbation direction by source coverage", REPRO / "Fig4" / "figure4b_delta_pcc_by_source_coverage.svg"),
    ("c", "Response-gene recovery by source coverage", REPRO / "Fig4" / "figure4c_top100_de_by_source_coverage.svg"),
    ("d", "Expression-range coverage by source coverage", REPRO / "Fig4" / "figure4d_erc_by_source_coverage.svg"),
    ("e", "Correlation structure by source coverage", REPRO / "Fig4" / "figure4e_csa_by_source_coverage.svg"),
    ("f", "Prior-interaction background ablation", REPRO / "Fig4" / "figure4f_prior_interaction_background.svg"),
]
assert all(path.exists() for _, _, path in outputs)
for panel, title, path in outputs:
    display(Markdown(f"### Fig. 4{panel}: {title}"))
    display(SVG(filename=str(path)))